In [1]:
from pxd.datasets.hdf5.directory import HDF5DirectoryDataset, HDF5DirectoryDatasetConfig
import numpy as np
import pandas as pd
from typing import Dict, Any, Tuple, List,Optional
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from samples.sample import Sample
import os
import json
import pickle

In [2]:
def get_samples(data_path):
    config = HDF5DirectoryDatasetConfig(
        path=data_path
    )

    dataset = HDF5DirectoryDataset(config)

    dataset_dict = dataset.load()

    samples = {}
    for i in tqdm(list(dataset_dict.keys())):
        samples[i] = dataset_dict[i]()
    
    return samples

In [9]:
train_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/train_predictions"
test_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/test_predictions"
val_pred_path="/flux/vault/stator/04_modelling/ds3_polynomial_ridge/validation_predictions"

In [4]:
train_samples = get_samples(train_pred_path)

  2%|▏         | 14/800 [00:00<00:05, 134.45it/s]

100%|██████████| 800/800 [00:05<00:00, 152.83it/s]


In [5]:
test_samples = get_samples(test_pred_path)

100%|██████████| 100/100 [00:00<00:00, 151.37it/s]


In [10]:
val_samples = get_samples(val_pred_path)

100%|██████████| 100/100 [00:00<00:00, 153.27it/s]


In [11]:


# ------------------------------
# Helper functions
# ------------------------------

def _ensure_1d(arr: Any, name: str = "array") -> np.ndarray:
    """Convert input to a 1D numpy array."""
    arr = np.asarray(arr, dtype=float)

    if arr.ndim == 2 and 1 in arr.shape:
        arr = arr.reshape(-1)

    if arr.ndim != 1:
        raise ValueError(f"Expected a 1D array for {name}, got shape {arr.shape}")

    return arr


def _ensure_2d(arr: Any, name: str = "array") -> np.ndarray:
    """Convert input to a 2D numpy array."""
    arr = np.asarray(arr, dtype=float)

    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D array for {name}, got shape {arr.shape}")

    return arr


def _get_first_available_key(dictionary: Dict[str, Any], key_candidates: List[str], sample_id: str) -> Any:
    """
    Allows flexible extraction if keys are named slightly differently.
    """
    for key in key_candidates:
        if key in dictionary:
            return dictionary[key]

    raise KeyError(
        f"None of the expected keys were found for sample '{sample_id}'. "
        f"Tried keys: {key_candidates}"
    )


def reconstruct_symmetric_matrix_from_upper_triangle(
    coeffs: Any,
    n: int = 16,
    diagonal_values: Optional[np.ndarray] = None,
    diagonal_strategy: str = "negative_row_sum"
) -> np.ndarray:
    """
    Reconstruct a full symmetric matrix from upper-triangle coefficients.

    Assumes coeffs were originally extracted using:
        a_matrix[np.triu_indices(n, k=1)]

    Parameters
    ----------
    coeffs : array-like
        Upper triangle coefficients excluding diagonal.
        Expected length = n * (n - 1) / 2.

    n : int
        Matrix size.

    diagonal_values : np.ndarray or None
        True diagonal values, if available.

    diagonal_strategy : str
        Options:
        - "from_true": use diagonal_values from the original a_matrix
        - "zero": set diagonal to zero
        - "negative_row_sum": set A_ii = -sum(A_ij for j != i)

    Returns
    -------
    a_matrix_pred : np.ndarray
        Reconstructed symmetric matrix, shape = (n, n)
    """
    coeffs = _ensure_1d(coeffs, name="a_matrix_coefficients_pred")

    expected_len = n * (n - 1) // 2
    if coeffs.shape[0] != expected_len:
        raise ValueError(
            f"Expected {expected_len} coefficients for upper triangle excluding diagonal, "
            f"got {coeffs.shape[0]}"
        )

    a_matrix_pred = np.zeros((n, n), dtype=float)

    upper_idx = np.triu_indices(n, k=1)

    a_matrix_pred[upper_idx] = coeffs
    a_matrix_pred = a_matrix_pred + a_matrix_pred.T

    if diagonal_strategy == "from_true":
        if diagonal_values is None:
            raise ValueError(
                "diagonal_strategy='from_true' requires diagonal_values from original a_matrix."
            )
        np.fill_diagonal(a_matrix_pred, diagonal_values)

    elif diagonal_strategy == "zero":
        np.fill_diagonal(a_matrix_pred, 0.0)

    elif diagonal_strategy == "negative_row_sum":
        diagonal = -np.sum(a_matrix_pred, axis=1)
        np.fill_diagonal(a_matrix_pred, diagonal)

    else:
        raise ValueError(
            "diagonal_strategy must be one of: 'from_true', 'zero', 'negative_row_sum'"
        )

    return a_matrix_pred


def calculate_metrics(y_true: Any, y_pred: Any, prefix: str = "") -> Dict[str, float]:
    """
    Calculate common regression metrics.

    Returns MAE, RMSE, relative RMSE, max absolute error, mean absolute percentage error,
    and R2 score.
    """
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"Shape mismatch for metrics: y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    error = y_pred - y_true
    abs_error = np.abs(error)

    mae = float(np.mean(abs_error))
    mse = float(np.mean(error ** 2))
    rmse = float(np.sqrt(mse))
    max_abs_error = float(np.max(abs_error))

    denominator = np.maximum(np.abs(y_true), 1e-12)
    mape = float(np.mean(abs_error / denominator) * 100)

    true_norm = float(np.linalg.norm(y_true))
    error_norm = float(np.linalg.norm(error))
    relative_l2_error = float(error_norm / max(true_norm, 1e-12))

    ss_res = float(np.sum(error ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else np.nan

    return {
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}relative_l2_error": relative_l2_error,
        f"{prefix}max_abs_error": max_abs_error,
        f"{prefix}mape_percent": mape,
        f"{prefix}r2": r2,
    }


def calculate_matrix_metrics(
    a_matrix_true: np.ndarray,
    a_matrix_pred: np.ndarray,
    prefix: str = "a_matrix_"
) -> Dict[str, float]:
    """
    Matrix-level metrics based on Frobenius norm and element-wise errors.
    """
    a_matrix_true = _ensure_2d(a_matrix_true, name="a_matrix_true")
    a_matrix_pred = _ensure_2d(a_matrix_pred, name="a_matrix_pred")

    if a_matrix_true.shape != a_matrix_pred.shape:
        raise ValueError(
            f"Shape mismatch: a_matrix_true={a_matrix_true.shape}, "
            f"a_matrix_pred={a_matrix_pred.shape}"
        )

    diff = a_matrix_pred - a_matrix_true

    fro_error = float(np.linalg.norm(diff, ord="fro"))
    fro_true = float(np.linalg.norm(a_matrix_true, ord="fro"))
    relative_fro_error = float(fro_error / max(fro_true, 1e-12))

    symmetry_error_pred = float(
        np.linalg.norm(a_matrix_pred - a_matrix_pred.T, ord="fro")
    )

    symmetry_error_true = float(
        np.linalg.norm(a_matrix_true - a_matrix_true.T, ord="fro")
    )

    metrics = {
        f"{prefix}frobenius_error": fro_error,
        f"{prefix}relative_frobenius_error": relative_fro_error,
        f"{prefix}symmetry_error_true": symmetry_error_true,
        f"{prefix}symmetry_error_pred": symmetry_error_pred,
    }

    metrics.update(
        calculate_metrics(
            a_matrix_true,
            a_matrix_pred,
            prefix=f"{prefix}elementwise_"
        )
    )

    return metrics


def build_coefficient_dataframe(
    sample_id: str,
    a_matrix_coefficients: np.ndarray,
    a_matrix_coefficients_pred: np.ndarray,
    n: int = 16
) -> pd.DataFrame:
    """
    Create long-format coefficient table for one sample.

    Each row corresponds to one upper-triangle matrix coefficient.
    """
    upper_i, upper_j = np.triu_indices(n, k=1)

    df = pd.DataFrame({
        "sample_id": sample_id,
        "matrix_i": upper_i,
        "matrix_j": upper_j,
        "a_matrix_coefficient_true": a_matrix_coefficients,
        "a_matrix_coefficient_pred": a_matrix_coefficients_pred,
    })

    df["a_matrix_coefficient_error"] = (
        df["a_matrix_coefficient_pred"] - df["a_matrix_coefficient_true"]
    )

    df["a_matrix_coefficient_abs_error"] = np.abs(
        df["a_matrix_coefficient_error"]
    )

    return df


# ------------------------------
# Main extraction, reconstruction, prediction and evaluation function
# ------------------------------

def build_and_evaluate_heatflux_predictions(
    samples_dict: Dict[str, Dict[str, Any]],
    n: int = 16,
    output_dir: str = "heatflux_evaluation_output",
    diagonal_strategy: str = "from_true"
) -> Tuple[Dict[str, Dict[str, np.ndarray]], pd.DataFrame, pd.DataFrame]:
    """
    Extract selected fields, reconstruct a_matrix_pred, calculate q_pred,
    evaluate performance, and save outputs.

    Extracted per sample:
      - a_matrix_coefficients
      - a_matrix_coefficients_pred
      - a_matrix
      - q_matrix
      - t_matrix

    Created per sample:
      - a_matrix_pred
      - q_pred
      - error matrices

    Parameters
    ----------
    samples_dict : dict
        Expected structure:
        {
            sample_id: {
                "data": {
                    "heatflux_matrix": {
                        "a_matrix_coefficients": ...,
                        "a_matrix_coefficients_pred": ...,
                        "a_matrix": ...,
                        "q_matrix": ...,
                        "t_matrix": ...
                    }
                }
            }
        }

    n : int, default=16
        Size of the square response matrix.

    output_dir : str
        Directory where outputs are saved.

    diagonal_strategy : str
        Strategy for rebuilding the diagonal of a_matrix_pred.
        Options:
        - "from_true": use diagonal from true a_matrix
        - "zero": set diagonal to zero
        - "negative_row_sum": infer diagonal using row-sum conservation

    Returns
    -------
    per_sample_results : dict
        Nested dictionary containing arrays per sample.

    metrics_df : pd.DataFrame
        One row per sample with evaluation metrics.

    coefficients_df : pd.DataFrame
        Long-format dataframe containing true/predicted coefficient values.
    """

    os.makedirs(output_dir, exist_ok=True)

    expected_coeff_len = n * (n - 1) // 2

    per_sample_results: Dict[str, Dict[str, np.ndarray]] = {}
    metrics_records: List[Dict[str, Any]] = []
    coefficient_dfs: List[pd.DataFrame] = []

    for sample_id, sample_payload in samples_dict.items():

        try:
            heatflux = sample_payload["data"]["heatflux_matrix"]
        except KeyError as e:
            raise KeyError(f"Missing key for sample '{sample_id}': {e}") from e

        # ------------------------------
        # Extract only required fields
        # ------------------------------

        a_matrix_coefficients = _ensure_1d(
            _get_first_available_key(
                heatflux,
                [
                    "a_matrix_coefficients",
                    "a_matrix_coefficient",
                    "a_matrix_coeffiecients"
                ],
                sample_id
            ),
            name=f"{sample_id}.a_matrix_coefficients"
        )

        a_matrix_coefficients_pred = _ensure_1d(
            _get_first_available_key(
                heatflux,
                [
                    "a_matrix_coefficients_pred",
                    "a_matrix_coefficient_pred",
                    "a_matrix_coefficients_prediction"
                ],
                sample_id
            ),
            name=f"{sample_id}.a_matrix_coefficients_pred"
        )

        a_matrix = _ensure_2d(
            _get_first_available_key(
                heatflux,
                ["a_matrix", "A_matrix", "response_matrix"],
                sample_id
            ),
            name=f"{sample_id}.a_matrix"
        )

        q_matrix = _ensure_2d(
            _get_first_available_key(
                heatflux,
                ["q_matrix", "Q_matrix", "heatflux_matrix_q"],
                sample_id
            ),
            name=f"{sample_id}.q_matrix"
        )

        t_matrix = _ensure_2d(
            _get_first_available_key(
                heatflux,
                ["t_matrix", "T_matrix", "temperature_matrix"],
                sample_id
            ),
            name=f"{sample_id}.t_matrix"
        )

        # ------------------------------
        # Validate shapes
        # ------------------------------

        if a_matrix.shape != (n, n):
            raise ValueError(
                f"Expected a_matrix shape ({n}, {n}) for sample '{sample_id}', "
                f"got {a_matrix.shape}"
            )

        if t_matrix.shape[0] != n:
            raise ValueError(
                f"Expected t_matrix to have {n} rows for sample '{sample_id}', "
                f"got {t_matrix.shape}"
            )

        if q_matrix.shape[0] != n:
            raise ValueError(
                f"Expected q_matrix to have {n} rows for sample '{sample_id}', "
                f"got {q_matrix.shape}"
            )

        if q_matrix.shape != (n, t_matrix.shape[1]):
            raise ValueError(
                f"Expected q_matrix shape ({n}, {t_matrix.shape[1]}) for sample '{sample_id}', "
                f"got {q_matrix.shape}"
            )

        if a_matrix_coefficients.shape[0] != expected_coeff_len:
            raise ValueError(
                f"Expected {expected_coeff_len} true coefficients for sample '{sample_id}', "
                f"got {a_matrix_coefficients.shape[0]}"
            )

        if a_matrix_coefficients_pred.shape[0] != expected_coeff_len:
            raise ValueError(
                f"Expected {expected_coeff_len} predicted coefficients for sample '{sample_id}', "
                f"got {a_matrix_coefficients_pred.shape[0]}"
            )

        # ------------------------------
        # Reconstruct predicted A matrix
        # ------------------------------

        diagonal_values = np.diag(a_matrix).copy()

        a_matrix_pred = reconstruct_symmetric_matrix_from_upper_triangle(
            coeffs=a_matrix_coefficients_pred,
            n=n,
            diagonal_values=diagonal_values,
            diagonal_strategy=diagonal_strategy
        )

        # ------------------------------
        # Calculate predicted heat flux
        # q_pred = A_pred @ T
        # ------------------------------

        q_pred = a_matrix_pred @ t_matrix

        # ------------------------------
        # Calculate errors
        # ------------------------------

        a_matrix_error = a_matrix_pred - a_matrix
        q_matrix_error = q_pred - q_matrix
        coefficient_error = a_matrix_coefficients_pred - a_matrix_coefficients

        # ------------------------------
        # Store per-sample arrays
        # ------------------------------

        per_sample_results[sample_id] = {
            "a_matrix_coefficients": a_matrix_coefficients,
            "a_matrix_coefficients_pred": a_matrix_coefficients_pred,
            "a_matrix_coefficients_error": coefficient_error,
            "a_matrix": a_matrix,
            "a_matrix_pred": a_matrix_pred,
            "a_matrix_error": a_matrix_error,
            "q_matrix": q_matrix,
            "q_pred": q_pred,
            "q_matrix_error": q_matrix_error,
            "t_matrix": t_matrix,
        }

        # ------------------------------
        # Metrics
        # ------------------------------

        sample_metrics: Dict[str, Any] = {
            "sample_id": sample_id,
            "n": n,
            "diagonal_strategy": diagonal_strategy,
            "num_coefficients": expected_coeff_len,
            "num_temperature_cases": t_matrix.shape[1],
        }

        sample_metrics.update(
            calculate_metrics(
                a_matrix_coefficients,
                a_matrix_coefficients_pred,
                prefix="coefficients_"
            )
        )

        sample_metrics.update(
            calculate_matrix_metrics(
                a_matrix,
                a_matrix_pred,
                prefix="a_matrix_"
            )
        )

        sample_metrics.update(
            calculate_metrics(
                q_matrix,
                q_pred,
                prefix="q_matrix_"
            )
        )

        metrics_records.append(sample_metrics)

        # ------------------------------
        # Long-format coefficient dataframe
        # ------------------------------

        coefficient_dfs.append(
            build_coefficient_dataframe(
                sample_id=sample_id,
                a_matrix_coefficients=a_matrix_coefficients,
                a_matrix_coefficients_pred=a_matrix_coefficients_pred,
                n=n
            )
        )

    # ------------------------------
    # Create analysis-friendly dataframes
    # ------------------------------

    metrics_df = pd.DataFrame(metrics_records)

    if coefficient_dfs:
        coefficients_df = pd.concat(coefficient_dfs, ignore_index=True)
    else:
        coefficients_df = pd.DataFrame()

    # ------------------------------
    # Global metrics across all samples
    # ------------------------------

    if per_sample_results:
        all_coeff_true = np.concatenate([
            result["a_matrix_coefficients"]
            for result in per_sample_results.values()
        ])

        all_coeff_pred = np.concatenate([
            result["a_matrix_coefficients_pred"]
            for result in per_sample_results.values()
        ])

        all_q_true = np.concatenate([
            result["q_matrix"].reshape(-1)
            for result in per_sample_results.values()
        ])

        all_q_pred = np.concatenate([
            result["q_pred"].reshape(-1)
            for result in per_sample_results.values()
        ])

        global_metrics = {
            "sample_id": "GLOBAL",
            "n": n,
            "diagonal_strategy": diagonal_strategy,
            "num_coefficients": len(all_coeff_true),
            "num_temperature_cases": np.nan,
        }

        global_metrics.update(
            calculate_metrics(
                all_coeff_true,
                all_coeff_pred,
                prefix="coefficients_"
            )
        )

        global_metrics.update(
            calculate_metrics(
                all_q_true,
                all_q_pred,
                prefix="q_matrix_"
            )
        )

        metrics_df = pd.concat(
            [metrics_df, pd.DataFrame([global_metrics])],
            ignore_index=True
        )

    # ------------------------------
    # Save outputs
    # ------------------------------

    # Best format for re-loading arrays exactly
    pickle_path = os.path.join(output_dir, "per_sample_results.pkl")
    with open(pickle_path, "wb") as f:
        pickle.dump(per_sample_results, f)

    # Human-readable summary metrics
    metrics_csv_path = os.path.join(output_dir, "evaluation_metrics.csv")
    metrics_df.to_csv(metrics_csv_path, index=False)

    # Long-format coefficient comparison
    coefficients_csv_path = os.path.join(output_dir, "coefficient_predictions_long_format.csv")
    coefficients_df.to_csv(coefficients_csv_path, index=False)

    # Also save compressed numpy archive for arrays
    npz_path = os.path.join(output_dir, "per_sample_arrays.npz")

    npz_save_dict = {}
    for sample_id, result in per_sample_results.items():
        safe_sample_id = str(sample_id).replace("/", "_").replace("\\", "_")

        for key, value in result.items():
            npz_save_dict[f"{safe_sample_id}__{key}"] = value

    np.savez_compressed(npz_path, **npz_save_dict)

    print("Saved outputs:")
    print(f"  Pickle arrays:      {pickle_path}")
    print(f"  Metrics CSV:        {metrics_csv_path}")
    print(f"  Coefficients CSV:   {coefficients_csv_path}")
    print(f"  Compressed NPZ:     {npz_path}")

    return per_sample_results, metrics_df, coefficients_df

In [12]:
per_sample_results_train, train_metrics_df, train_coefficients_df = build_and_evaluate_heatflux_predictions(samples_dict=train_samples,
                                                                                        n=16,
                                                                                        output_dir="output/botorch_train",
                                                                                        diagonal_strategy="from_true"
                                                                                        )
                                                                                            
train_metrics_df.head()

KeyError: "None of the expected keys were found for sample 'sample_732'. Tried keys: ['a_matrix', 'A_matrix', 'response_matrix']"

In [13]:
"""clean_data = {
    key_1: { val_1: (val_2.tolist() if isinstance(val_2, np.ndarray) else val)} 
    for keys in per_sample_results_train.keys()
}
"""
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)
        
with open("output/botorch_train/per_sample_results_train.json", "w") as f:
        json.dump(per_sample_results_train, f, sort_keys=True, cls=NumpyEncoder)

NameError: name 'per_sample_results_train' is not defined

In [14]:
per_sample_results_test, test_metrics_df, test_coefficients_df = build_and_evaluate_heatflux_predictions(samples_dict=test_samples,
                                                                                        n=16,
                                                                                        output_dir="output/botorch_test",
                                                                                        diagonal_strategy="from_true"
                                                                                        )
                                                                                            
test_metrics_df.head()

KeyError: "None of the expected keys were found for sample 'sample_365'. Tried keys: ['a_matrix', 'A_matrix', 'response_matrix']"

In [19]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)
        
with open("output/botorch_train/per_sample_results_test.json", "w") as f:
        json.dump(per_sample_results_test, f, sort_keys=True, cls=NumpyEncoder)

In [15]:
per_sample_results_val, val_metrics_df, val_coefficients_df = build_and_evaluate_heatflux_predictions(samples_dict=val_samples,
                                                                                        n=16,
                                                                                        output_dir="output/botorch_val",
                                                                                        diagonal_strategy="from_true"
                                                                                        )
                                                                                            
val_metrics_df.head()

KeyError: "None of the expected keys were found for sample 'sample_802'. Tried keys: ['a_matrix', 'A_matrix', 'response_matrix']"

In [21]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)
        
with open("output/botorch_train/per_sample_results_val.json", "w") as f:
        json.dump(per_sample_results_val, f, sort_keys=True, cls=NumpyEncoder)